In [42]:
#1
!pip install -q tensorflow

#  Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
#2
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Conv1D, GlobalAveragePooling1D
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split


np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print(f" TensorFlow version: {tf.__version__}")
print(f" GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

 TensorFlow version: 2.20.0
 GPU available: False


In [44]:
#3
BASE_PATH = '/content/drive/MyDrive/GP/CSV_completeDataset'

def load_csvs(base_path, split):
    all_dfs = []
    for class_name, label in [('Alert', 0), ('Drowsy', 1)]:
        folder = os.path.join(base_path, split, class_name)
        files = glob.glob(os.path.join(folder, '*.csv'))

        for f in files:
            df = pd.read_csv(f)
            filename   = os.path.basename(f).replace('.csv', '')
            video_id   = filename
            subject_id = filename.split('_')[0]

            df['video_id']   = video_id
            df['subject_id'] = subject_id
            df['label']      = label
            df['class_name'] = class_name
            all_dfs.append(df)

    return pd.concat(all_dfs, ignore_index=True)

scaler = StandardScaler()
train_df = load_csvs(BASE_PATH, 'training')
valid_df = load_csvs(BASE_PATH, 'valid')
test_df  = load_csvs(BASE_PATH, 'testing')


feature_cols = ['Duration', 'Amplitude', 'Velocity', 'Frequency']

# Fit scaler ONLY on training data
scaler.fit(train_df[feature_cols])

# Transform all datasets
train_df[feature_cols] = scaler.transform(train_df[feature_cols])
valid_df[feature_cols] = scaler.transform(valid_df[feature_cols])
test_df[feature_cols]  = scaler.transform(test_df[feature_cols])

print("=" * 50)

print(f"Training data: ")
print(f"   Number of videos: {train_df['video_id'].nunique()}")
print(f"   Total blinks: {len(train_df)}")

print(f"\nValidation data:")
print(f"   Number of videos: {valid_df['video_id'].nunique()}")
print(f"   Total blinks: {len(valid_df)}")

print(f"\nTest data:")
print(f"   Number of videos: {test_df['video_id'].nunique()}")
print(f"   Total blinks: {len(test_df)}")

Training data: 
   Number of videos: 52
   Total blinks: 4048

Validation data:
   Number of videos: 10
   Total blinks: 740

Test data:
   Number of videos: 10
   Total blinks: 1342


In [45]:
#4
#defult if I don't choose a value for them
def build_window_features(df, window_size=8, stride=2):


#Builds windows using only raw values ​​(4 features) Suitable for LSTM

    feature_cols = ['Duration', 'Amplitude', 'Velocity', 'Frequency']
    rows = []

    for video_id, g in df.groupby('video_id'):
        g = g.sort_values('Blink_ID').reset_index(drop=True)

        if len(g) < window_size:
            continue

        label      = g['label'].iloc[0]
        subject_id = g['subject_id'].iloc[0]

        for start in range(0, len(g) - window_size + 1, stride):
            window = g.iloc[start:start + window_size]

            feat = {
                'video_id': video_id,
                'subject_id': subject_id,
                'label': label,
                'window_idx': start,

                'data': window[feature_cols].values
            }

            rows.append(feat)

    return pd.DataFrame(rows)


# ================================
WINDOW_SIZE = 8
STRIDE = 2

print(" Building training windows...")
train_windows = build_window_features(train_df, WINDOW_SIZE, STRIDE)
print(f" Training windows: {len(train_windows)}")

print("\n Building validation windows...")
valid_windows = build_window_features(valid_df, WINDOW_SIZE, STRIDE)
print(f" validation windows: {len(valid_windows)}")

print("\n Building test windows...")
test_windows = build_window_features(test_df, WINDOW_SIZE, STRIDE)
print(f" Test windows: {len(test_windows)}")


feature_cols = ['Duration', 'Amplitude', 'Velocity', 'Frequency']
print(f"\n Number of features per window: {len(feature_cols)}")

 Building training windows...
 Training windows: 1856

 Building validation windows...
 validation windows: 337

 Building test windows...
 Test windows: 637

 Number of features per window: 4


In [46]:
def windows_to_sequences(windows_df, sequence_length=5):
    sequences = []
    labels = []
    video_ids = []
    subject_ids = []

    for video_id, g in windows_df.groupby('video_id'):
        g = g.sort_values('window_idx').reset_index(drop=True)

        if len(g) < sequence_length:
            continue

        for i in range(len(g) - sequence_length + 1):

            window_seq = g.iloc[i:i + sequence_length]['data'].tolist()

            window_seq = [w.reshape(-1) for w in window_seq]

            seq = np.array(window_seq, dtype=np.float32)

            sequences.append(seq)
            labels.append(g['label'].iloc[i])
            video_ids.append(video_id)
            subject_ids.append(g['subject_id'].iloc[i])

    return (np.array(sequences, dtype=np.float32),
            np.array(labels),
            np.array(video_ids),
            np.array(subject_ids))


# =================================
SEQUENCE_LENGTH = 5


print(f"Convert windows to sequences {SEQUENCE_LENGTH}")
print()

X_train_seq, y_train, train_video_ids, _ = windows_to_sequences(
    train_windows, SEQUENCE_LENGTH)

X_valid_seq, y_valid, valid_video_ids, _ = windows_to_sequences(
    valid_windows, SEQUENCE_LENGTH)

X_test_seq, y_test, test_video_ids, _ = windows_to_sequences(
    test_windows, SEQUENCE_LENGTH)


#(Samples, Time Steps, Features)

print(f" Training data shape:  {X_train_seq.shape}")
print(f" Test data shape: {X_test_seq.shape}")

print(f"\n Class distribution in training:")
print(f"   Alert:  {(y_train == 0).sum()}")
print(f"   Drowsy: {(y_train == 1).sum()}")

Convert windows to sequences 5

 Training data shape:  (1654, 5, 32)
 Test data shape: (597, 5, 32)

 Class distribution in training:
   Alert:  722
   Drowsy: 932


In [47]:

input_shape = (X_train_seq.shape[1], X_train_seq.shape[2])
gru_model = Sequential([
    # First GRU layer tracking transitions between your 5 windows
    GRU(128, return_sequences=True, input_shape=input_shape),
    BatchNormalization(), # Keeps features scaled across your sequences
    Dropout(0.3),

    # Second GRU layer summarizing the whole sequence into a state vector
    GRU(64, return_sequences=False),
    BatchNormalization(),
    Dropout(0.3),

    # Dense interpretation layer
    Dense(32, activation='relu'),
    Dropout(0.2),

    # Final classification layer (Binary: Drowsy vs. Alert)
    Dense(1, activation='sigmoid')
])

# 3. Compile the model
gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

gru_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_8 (GRU)                     │ (None, 5, 128)         │        62,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 5, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 5, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_9 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 102,337 (399.75 KB)

 Trainable params: 101,953 (398.25 KB)

 Non-trainable params: 384 (1.50 KB)

In [53]:
history = gru_model.fit(
    X_train_seq, y_train,
    validation_data=(X_valid_seq, y_valid),
    epochs=30,
    batch_size=32
)

Epoch 1/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9601 - loss: 0.1114 - val_accuracy: 0.8215 - val_loss: 0.7473
Epoch 2/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.9577 - loss: 0.1079 - val_accuracy: 0.8215 - val_loss: 0.7587
Epoch 3/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.9595 - loss: 0.1138 - val_accuracy: 0.8182 - val_loss: 0.7765
Epoch 4/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9534 - loss: 0.1202 - val_accuracy: 0.8182 - val_loss: 0.7765
Epoch 5/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9601 - loss: 0.1082 - val_accuracy: 0.8182 - val_loss: 0.7806
Epoch 6/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9601 - loss: 0.1018 - val_accuracy: 0.8148 - val_loss: 0.7860
Epoch 7/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9637 - loss: 0.0984 - val_accuracy: 0.8114 - val_loss: 0.8024
Epoch 8/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9637 - loss: 0.0980 - val_accuracy: 0.8148 - v

In [49]:
def evaluate_hierarchical_model_by_file(model, model_name, test_base, window_size=8, stride=2, sequence_length=5):
    total_sequences_tested = 0
    correct_sequences = 0
    total_videos_tested = 0
    correct_videos = 0

    print(f"\n========== {model_name} RESULTS ==========")
    print(f"{'File Name':<25} | {'Label':<8} | {'Seq Acc':<12} | {'Drowsy %':<10} | {'Verdict':<8} | {'Status'}")
    print("-" * 95)

    for category in ['Drowsy', 'Alert']:
        folder_path = os.path.join(test_base, category)

        if not os.path.exists(folder_path):
            print(f"Warning: Folder not found: {folder_path}")
            continue

        true_val = 1 if category == 'Drowsy' else 0

        for filename in os.listdir(folder_path):
            if filename.endswith(".csv"):
                file_path = os.path.join(folder_path, filename)
                df_file = pd.read_csv(file_path)

                # Chronological sort step
                if 'Blink_ID' in df_file.columns:
                    df_file = df_file.sort_values('Blink_ID').reset_index(drop=True)

                feats = df_file[['Duration', 'Amplitude', 'Velocity', 'Frequency']].values

                # Step 1: Scale raw features safely before slicing
                feats_scaled = feats

                if len(feats_scaled) < window_size:
                    continue

                # Step 2: Build the individual windows
                file_windows = []
                for i in range(0, len(feats_scaled) - window_size + 1, stride):
                    file_windows.append(feats_scaled[i:i + window_size])

                if len(file_windows) < sequence_length:
                    continue

                # Step 3: Combine windows into sliding window sequences
                file_sequences = []
                for i in range(len(file_windows) - sequence_length + 1):
                    window_seq = file_windows[i:i + sequence_length]
                    # Matrix flatten: converts each (8, 4) window inside the sequence into (32,)
                    flattened_seq = [w.reshape(-1) for w in window_seq]
                    file_sequences.append(flattened_seq)

                # Array shape is now (Samples, 5, 32)
                X_file_final = np.array(file_sequences, dtype=np.float32)

                # Matches the exact input structure the GRU was trained on
                file_preds = model.predict(X_file_final, verbose=0)

                # Fix 2: Explicitly turn probabilities into 0 or 1 classifications
                file_rounded = (file_preds > 0.5).astype(int).flatten()

                # Sequence accuracy for this file
                file_seq_acc = np.mean(file_rounded == true_val) * 100

                total_sequences_tested += len(file_rounded)
                correct_sequences += np.sum(file_rounded == true_val)

                # Video-level decision
                drowsy_percent = (np.sum(file_rounded) / len(file_rounded)) * 100
                verdict_val = 1 if drowsy_percent > 50 else 0

                total_videos_tested += 1

                if verdict_val == true_val:
                    correct_videos += 1

                status = "✅" if verdict_val == true_val else "❌"
                verdict_text = "DROWSY" if verdict_val == 1 else "ALERT"

                print(
                    f"{filename[:25]:<25} | "
                    f"{category:<8} | "
                    f"{file_seq_acc:>9.1f}% | "
                    f"{drowsy_percent:>8.1f}% | "
                    f"{verdict_text:<8} | "
                    f"{status}"
                )

    sequence_accuracy = (correct_sequences / total_sequences_tested) * 100 if total_sequences_tested > 0 else 0
    video_accuracy = (correct_videos / total_videos_tested) * 100 if total_videos_tested > 0 else 0

    print("\n" + "=" * 55)
    print(f"{model_name} TOTAL SEQUENCE ACCURACY: {sequence_accuracy:.2f}%")
    print(f"{model_name} TOTAL VIDEO ACCURACY:    {video_accuracy:.2f}%")
    print("=" * 55)

    return sequence_accuracy, video_accuracy

In [50]:
test_folder_path = os.path.join(BASE_PATH, 'testing')

gru_seq_acc, gru_video_acc = evaluate_hierarchical_model_by_file(
    model=gru_model,
    model_name="Hierarchical GRU",
    test_base=test_folder_path,
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    sequence_length=SEQUENCE_LENGTH
)


========== Hierarchical GRU RESULTS ==========
File Name                 | Label    | Seq Acc      | Drowsy %   | Verdict  | Status
-----------------------------------------------------------------------------------------------
D022_20260513_190626_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
D023_20260513_195650_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
D024_20260513_212814_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
D025_20260513_233342_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
D026_20260514_025307_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
A022_20260513_190541_fram | Alert    |       0.0% |    100.0% | DROWSY   | ❌
A023_20260513_195602_fram | Alert    |       0.0% |    100.0% | DROWSY   | ❌
A024_20260513_212736_fram | Alert    |       0.0% |    100.0% | DROWSY   | ❌
A025_20260513_233235_fram | Alert    |       0.0% |    100.0% | DROWSY   | ❌
A026_20260514_025227_fram | Alert    |       0.0% |    100.0% | DROWSY   | ❌



TCN

In [56]:
input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])
tcn_model = Sequential([
    Conv1D(64, kernel_size=3, dilation_rate=1, padding='causal', activation='relu', input_shape=input_shape),
    Dropout(0.3),

    Conv1D(64, kernel_size=3, dilation_rate=2, padding='causal', activation='relu'),
    Dropout(0.3),

    Conv1D(32, kernel_size=3, dilation_rate=4, padding='causal', activation='relu'),
    Dropout(0.3),

    GlobalAveragePooling1D(),

    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

tcn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

tcn_model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_6 (Conv1D)               │ (None, 5, 64)          │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 5, 64)          │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_22 (Dropout)            │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 5, 32)          │         6,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_23 (Dropout)            │ (None, 5, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,281 (98.75 KB)

 Trainable params: 25,281 (98.75 KB)

 Non-trainable params: 0 (0.00 B)

In [57]:
tcn_history = tcn_model.fit(
    X_train_seq, y_train,
    validation_data=(X_valid_seq, y_valid),
    epochs=30,
    batch_size=32
)

#tcn_save_path = '/content/drive/MyDrive/GP/LstmModels/best_tcn_drowsy_model.h5'
#tcn_model.save(tcn_save_path)

#print(f"🚀 TCN model saved successfully to: {tcn_save_path}")

Epoch 1/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.4559 - loss: 0.6972 - val_accuracy: 0.5825 - val_loss: 0.6888
Epoch 2/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5877 - loss: 0.6851 - val_accuracy: 0.7845 - val_loss: 0.6797
Epoch 3/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7243 - loss: 0.6695 - val_accuracy: 0.7710 - val_loss: 0.6621
Epoch 4/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7908 - loss: 0.6376 - val_accuracy: 0.7744 - val_loss: 0.6271
Epoch 5/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8235 - loss: 0.5781 - val_accuracy: 0.7811 - val_loss: 0.5681
Epoch 6/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8416 - loss: 0.4858 - val_accuracy: 0.7879 - val_loss: 0.4987
Epoch 7/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8640 - loss: 0.3963 - val_accuracy: 0.7946 - val_loss: 0.4552
Epoch 8/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8839 - loss: 0.3284 - val_accuracy: 0.8047 - val_

In [61]:
test_folder_path = os.path.join(BASE_PATH, 'testing')
tcn_window_acc, tcn_video_acc = evaluate_hierarchical_model_by_file(
    tcn_model,
    "TCN",
    test_folder_path,
    WINDOW_SIZE,
    STRIDE
)


========== TCN RESULTS ==========
File Name                 | Label    | Seq Acc      | Drowsy %   | Verdict  | Status
-----------------------------------------------------------------------------------------------
D022_20260513_190626_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
D023_20260513_195650_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
D024_20260513_212814_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
D025_20260513_233342_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
D026_20260514_025307_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
A022_20260513_190541_fram | Alert    |       0.0% |    100.0% | DROWSY   | ❌
A023_20260513_195602_fram | Alert    |       0.0% |    100.0% | DROWSY   | ❌
A024_20260513_212736_fram | Alert    |       0.0% |    100.0% | DROWSY   | ❌
A025_20260513_233235_fram | Alert    |       0.0% |    100.0% | DROWSY   | ❌
A026_20260514_025227_fram | Alert    |       0.0% |    100.0% | DROWSY   | ❌

TCN TOTAL SEQ